<a href="https://colab.research.google.com/github/frevyorencia/objectdetectionbananaapple/blob/main/applebanana_data_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install datasets ultralytics

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
from datasets import load_dataset
from PIL import Image
from pathlib import Path

SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    save_root = Path("/content/drive/MyDrive/apple_banana_dataset")
else:
    save_root = Path("/content/apple_banana_dataset")

(save_root / "apple" / "images").mkdir(parents=True, exist_ok=True)
(save_root / "apple" / "labels").mkdir(parents=True, exist_ok=True)
(save_root / "banana" / "images").mkdir(parents=True, exist_ok=True)
(save_root / "banana" / "labels").mkdir(parents=True, exist_ok=True)

print("Downloading dataset...")
dataset = load_dataset("JadeRay-42/Friut-Detection", split="train")

apple_count = 0
banana_count = 0

for idx, sample in enumerate(dataset):
    categories = sample["objects"]["category"]

    if 0 in categories:
        fruit_type = "apple"
        apple_count += 1
    elif 1 in categories:
        fruit_type = "banana"
        banana_count += 1
    else:
        continue

    image = sample["image"]
    img_filename = f"{fruit_type}_{apple_count if fruit_type=='apple' else banana_count}.jpg"
    img_path = save_root / fruit_type / "images" / img_filename
    image.save(img_path)

    img_w, img_h = image.size
    label_filename = f"{fruit_type}_{apple_count if fruit_type=='apple' else banana_count}.txt"
    label_path = save_root / fruit_type / "labels" / label_filename

    with open(label_path, "w") as f:
        for bbox, cat in zip(sample["objects"]["bbox"], categories):
            if (fruit_type == "apple" and cat == 0) or (fruit_type == "banana" and cat == 1):
                x_min, y_min, x_max, y_max = bbox
                x_center = (x_min + x_max) / 2 / img_w
                y_center = (y_min + y_max) / 2 / img_h
                width = (x_max - x_min) / img_w
                height = (y_max - y_min) / img_h
                class_id = 0 if fruit_type == "apple" else 1
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    if (idx + 1) % 20 == 0:
        print(f"Processed {idx + 1} images... (Apples: {apple_count}, Bananas: {banana_count})")

print(f"\n✅ Complete! Saved {apple_count} apples and {banana_count} bananas")

Processed 20 images... (Apples: 20, Bananas: 0)
Processed 40 images... (Apples: 40, Bananas: 0)
Processed 60 images... (Apples: 60, Bananas: 0)
Processed 80 images... (Apples: 75, Bananas: 5)
Processed 100 images... (Apples: 75, Bananas: 25)
Processed 120 images... (Apples: 75, Bananas: 45)
Processed 140 images... (Apples: 75, Bananas: 65)
Processed 160 images... (Apples: 84, Bananas: 76)

✅ Complete! Saved 90 apples and 77 bananas


In [24]:
import shutil
import random
from pathlib import Path

if SAVE_TO_DRIVE:
    source_root = Path("/content/drive/MyDrive/apple_banana_dataset")
    yolo_root = Path("/content/drive/MyDrive/yolov8_dataset")
else:
    source_root = Path("/content/apple_banana_dataset")
    yolo_root = Path("/content/yolov8_dataset")

# Create YOLOv8 structure
(yolo_root / "images" / "train").mkdir(parents=True, exist_ok=True)
(yolo_root / "images" / "val").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "train").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "val").mkdir(parents=True, exist_ok=True)

# Collect all files
all_files = []
for fruit in ["apple", "banana"]:
    img_dir = source_root / fruit / "images"
    label_dir = source_root / fruit / "labels"

    if img_dir.exists():
        for img_file in img_dir.glob("*.jpg"):
            label_file = label_dir / f"{img_file.stem}.txt"
            if label_file.exists():
                all_files.append((img_file, label_file))

print(f"Found {len(all_files)} total images with labels")

# Split and copy
random.seed(42)
random.shuffle(all_files)
split_idx = int(len(all_files) * 0.8)
train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

for img_file, label_file in train_files:
    shutil.copy(img_file, yolo_root / "images" / "train" / img_file.name)
    shutil.copy(label_file, yolo_root / "labels" / "train" / label_file.name)

for img_file, label_file in val_files:
    shutil.copy(img_file, yolo_root / "images" / "val" / img_file.name)
    shutil.copy(label_file, yolo_root / "labels" / "val" / label_file.name)

# Create data.yaml
data_yaml = f"""# YOLOv8 Dataset Configuration
path: {yolo_root.absolute()}
train: images/train
val: images/val

nc: 2
names: ['apple', 'banana']
"""

with open(yolo_root / "data.yaml", "w") as f:
    f.write(data_yaml.strip())

print(f"✅ Dataset organized at {yolo_root}")
print(f"   Train: {len(train_files)} images")
print(f"   Val: {len(val_files)} images")

Found 167 total images with labels
✅ Dataset organized at /content/drive/MyDrive/yolov8_dataset
   Train: 133 images
   Val: 34 images


In [27]:
from ultralytics import YOLO
import torch

# Check available device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

dataset_yaml = "/content/drive/MyDrive/yolov8_dataset/data.yaml"

model = YOLO('yolov8n.pt')

results = model.train(
    data=dataset_yaml,
    epochs=10,
    imgsz=640,
    batch=8,                    # Reduced batch size for CPU
    device=device,              # Will be 'cpu' in your case
    workers=4,                  # Reduced workers for CPU
    project='/content/drive/MyDrive/yolov8_trained_model',
    name='exp1',
    exist_ok=True
)

print("✅ Training complete!")

Using device: cpu
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolov8_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto,

In [28]:
from google.colab import files
files.download('/content/drive/MyDrive/yolov8_trained_model/exp1/weights/best.pt')
print("✅ Model downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model downloaded!
